# Agents — type routing + name parsing

Runs the rungs of `mds_norm.pipeline.persons` one at a time so each can be measured: a regex type router (person / people / organisation / multiple / residue), a deterministic parser for inverted forms, the probablepeople CRF for the natural-order remainder with a copy-check gate, and deferral rates per institution. ULAN linking runs afterwards in `mds_norm.pipeline.agent_links`.

In [1]:
import polars as pl

from mds_norm.parsers.parse_person import parse_person
from mds_norm.paths import FIELD_STATS, PERSON_ANNOTATIONS, PERSON_DECISIONS, PERSONS_OUT
from mds_norm.pipeline import persons

In [2]:
# Runs the persons stage rung by rung, for measurement
PERSONS_OUT.mkdir(parents=True, exist_ok=True)
print(f"{len(persons.AGENT_FIELDS)} agent fields, {len(persons.ORG_ONLY_FIELDS)} of them organisation-only")

26 agent fields, 4 of them organisation-only


## 1. Distinct person values

Dedupe on distinct value; decisions join back to `(record_id, node_id)` at the end.

In [3]:
cells = persons.agent_cells(pl.scan_parquet(FIELD_STATS))
routed = persons.route(cells)

print(f"{len(routed)} distinct values / {routed['count'].sum()} occurrences")
routed.select("value", "count", "n_institutions").head()

542661 distinct values / 5451672 occurrences


value,count,n_institutions
str,u32,u32
"""Petters Ltd""",399264,3
"""Unknown""",245601,40
"""Belliss & Morcom Ltd""",97500,1
"""Davey, Paxman & Co. Ltd.""",59293,1
"""R.A. Lister & Co. Ltd""",49795,2


## 2. Type router

Mixed content defers to the residue queue before any type is assigned; multi-entity values defer to atomisation.

In [4]:
# Bootstrap bucket regexes, to be replaced by induced ones
routed.group_by("route").agg(distinct=pl.len(), occurrences=pl.col("count").sum()).sort(
    "occurrences", descending=True
)

route,distinct,occurrences
str,u32,u32
"""person""",395179,3172063
"""organisation""",65401,1313952
"""residue""",71081,594088
"""knowledge_state""",17,293416
"""placeholder""",16,38176
"""multiple""",9145,27143
"""people""",1822,12834


## 3. Person parsers

`mds_norm.parsers.parse_person` tries the conventions in precision order and falls back to the CRF: single-comma inverted form with particle reattachment, the trailing-title form (closed sets only), Nottingham's slash/angle convention, then probablepeople with a particle rule layer and an alpha-token copy check that turns parse errors into deferrals. `Corporation` tags and non-person coordinations (`Harland and Wolff`) are emitted as organisations.

In [5]:
# Sub-component names are the review strata, so rungs demote themselves
for example in ["Dürer, Albrecht", "White, Francis Buchanan, Dr", "Reedie, Kenneth, Mr, MBE",
                "Julian Carr, Woodbridge, Suffolk", "Carr, J.W. < M.A.", "Carr/J.W.<M.A",
                "Harland and Wolff", "Gurney, Sophie and Hambro, Elisabeth", "DCM"]:
    name, entity, sub, reason = parse_person(example)
    parts = {k: v for k, v in (name or {}).items() if v and k != "display"}
    print(f"{example!r:40} {entity:12} {sub:20} {reason or parts}")

'Dürer, Albrecht'                        person       inverted             {'given': 'Albrecht', 'surname': 'Dürer'}
'White, Francis Buchanan, Dr'            person       inverted_tail        {'prefix': 'Dr', 'given': 'Francis', 'middle': 'Buchanan', 'surname': 'White'}
'Reedie, Kenneth, Mr, MBE'               person       inverted_tail        {'prefix': 'Mr', 'given': 'Kenneth', 'surname': 'Reedie', 'suffix': 'MBE'}
'Julian Carr, Woodbridge, Suffolk'       person       crf                  repeated_component:given
'Carr, J.W. < M.A.'                      person       angle_tail           {'given': 'J.W.', 'surname': 'Carr', 'suffix': 'M.A.'}
'Carr/J.W.<M.A'                          person       slashed              {'given': 'J.W.', 'surname': 'Carr', 'suffix': 'M.A'}
'Harland and Wolff'                      organisation crf_and_organisation {}
'Gurney, Sophie and Hambro, Elisabeth'   person       crf                  unmapped_label:And
'DCM'                                    person 

In [6]:
person_decisions = persons.parse_person_route(routed)

person_decisions.group_by("sub_component", pl.col("defer_reason").is_null().alias("parsed")).len()

sub_component,parsed,len
str,bool,u32
"""crf""",false,27777
"""crf""",true,98038
"""crf_and_organisation""",true,1120
"""slashed""",false,2828
"""slashed""",true,15600
"""crf_corporation""",true,48470
"""angle_tail""",true,63
"""inverted""",true,188201
"""inverted_tail""",true,13082


## 4. Organisation / people routes

Tier 2 resolves the type and keeps the value whole. Multiple/residue routes defer with the route as reason.

In [7]:
# Router keeps the value whole and defers multiple or residue
decisions = persons.value_decisions(routed)
decisions.write_parquet(PERSON_DECISIONS)
print(len(decisions), "value decisions ->", PERSON_DECISIONS)

decisions.head()

542661 value decisions -> /home/liam/Documents/university/mds-norm/data/analysis_output/persons/person_value_decisions.parquet


value,entity_type,sub_component,defer_reason,prefix,given,middle,nickname,surname,suffix,display,count,component,tier,status,confidence
str,str,str,str,str,str,str,str,str,str,str,u32,str,i32,str,f64
"""Petters Ltd""","""organisation""","""router""",null,null,null,null,null,null,null,"""Petters Ltd""",399264,"""persons""",2,"""resolved""",0.8
"""Unknown""","""knowledge_state""","""router""","""knowledge_state""",null,null,null,null,null,null,null,245601,"""persons""",2,"""deferred""",null
"""Belliss & Morcom Ltd""","""organisation""","""router""",null,null,null,null,null,null,null,"""Belliss & Morcom Ltd""",97500,"""persons""",2,"""resolved""",0.8
"""Davey, Paxman & Co. Ltd.""","""organisation""","""router""",null,null,null,null,null,null,null,"""Davey, Paxman & Co. Ltd.""",59293,"""persons""",2,"""resolved""",0.8
"""R.A. Lister & Co. Ltd""","""organisation""","""router""",null,null,null,null,null,null,null,"""R.A. Lister & Co. Ltd""",49795,"""persons""",2,"""resolved""",0.8


### 4a. Mononyms, typed by the field's own contract

A lone capitalised word has no slot in the Person model, but in an organisation-only field it is taken as recorded. This rung is keyed by `(value, field)`.

In [8]:
mononym_decisions = persons.mononym_decisions(cells, decisions)

print(f"{len(mononym_decisions):,} (value, organisation-only field) pairs read as a mononym")
mononym_decisions.head()

1,354 (value, organisation-only field) pairs read as a mononym


value,field_type,entity_type_field,display_field,sub_component_field,confidence_field
str,str,str,str,str,f64
"""Chantilly""","""spectrum/object_production_org…","""organisation""","""Chantilly""","""mononym_organisation""",0.8
"""Lunds""","""spectrum/object_production_org…","""organisation""","""Lunds""","""mononym_organisation""",0.8
"""Bukta""","""spectrum/object_production_org…","""organisation""","""Bukta""","""mononym_organisation""",0.8
"""Annacat""","""spectrum/object_production_org…","""organisation""","""Annacat""","""mononym_organisation""",0.8
"""Calor""","""spectrum/object_production_org…","""organisation""","""Calor""","""mononym_organisation""",0.8


## 5. Sidecar assembly

Decisions join by value onto every occurrence; spans cover the whole value.

In [9]:
annotations = persons.annotations(cells, decisions, mononym_decisions)
annotations.write_parquet(PERSON_ANNOTATIONS)
print(len(annotations), "annotation rows ->", PERSON_ANNOTATIONS)

5451672 annotation rows -> /home/liam/Documents/university/mds-norm/data/analysis_output/persons/person_annotations.parquet


## 6. Deferral rates

Occurrence- and distinct-weighted per institution, then residue grouped by induced shape as the worklist.

In [10]:
per_institution = (
    annotations
    .group_by("data_source")
    .agg(
        occurrences=pl.len(),
        deferred_occ=(pl.col("status") == "deferred").mean(),
    )
    .join(
        annotations.unique(["data_source", "value"])
        .group_by("data_source")
        .agg(distinct=pl.len(), deferred_distinct=(pl.col("status") == "deferred").mean()),
        on="data_source",
    )
    .sort("deferred_occ", descending=True)
)

overall_occ = (annotations["status"] == "deferred").mean()
overall_distinct = (decisions["status"] == "deferred").mean()
print(f"deferred: {overall_occ:.1%} of occurrences, {overall_distinct:.1%} of distinct values")
per_institution

deferred: 22.2% of occurrences, 20.4% of distinct values


data_source,occurrences,deferred_occ,distinct,deferred_distinct
enum,u32,f64,u32,f64
"""Live Borders""",421,0.997625,12,0.916667
"""Armagh Observatory & Planetari…",15792,0.995187,921,0.988056
"""Cultural Collections & Galleri…",3365,0.989302,922,0.972885
"""Royal Albert Memorial Museum &…",19278,0.989159,2695,0.952876
"""Wotton-under-Edge Heritage Cen…",14403,0.861001,686,0.14723
…,…,…,…,…
"""Southampton Cultural Services""",21114,0.000332,42,0.047619
"""Bristol Museums""",10,0.0,7,0.0
"""Torquay Museum""",14,0.0,7,0.0


In [11]:
residue_worklist = (
    decisions.filter(pl.col("status") == "deferred")
    .with_columns(
        shape=pl.col("value").str.replace_all(r"[A-Za-z]+", "s").str.replace_all(r"\d+", "d"))
    .group_by("shape", "defer_reason")
    .agg(distinct=pl.len(), occurrences=pl.col("count").sum(),
         examples=pl.col("value").head(3))
    .sort("occurrences", descending=True)
)
residue_worklist.head(20)

shape,defer_reason,distinct,occurrences,examples
str,str,u32,u32,list[str]
"""s""","""knowledge_state""",9,292436,"[""Unknown"", ""unknown"", ""Unidentified""]"
"""s""","""no_surname""",6849,86107,"[""Wolseley"", ""Victoria"", ""Chadwick""]"
"""s""","""unmapped_label:ShortForm""",1239,65548,"[""DCM"", ""S"", ""JWC""]"
"""s s""","""residue""",128,47112,"[""Unknown maker"", ""Anonymous British"", ""Unknown photographer""]"
"""s""","""placeholder""",8,35434,"[""NULL"", ""Various"", ""various""]"
…,…,…,…,…
"""s s (s) s""","""residue""",185,9863,"[""Terence Soames (Cardiff) Ltd"", ""John Fowler (Leeds) Ltd"", ""Oil Engines (Coventry) Ltd""]"
"""s s s""","""no_surname""",49,9292,"[""King Edward I"", ""Queen Elizabeth II"", ""Queen Elizabeth I""]"
"""s s (s)""","""residue""",1804,9001,"[""Boyden Observatory (cre)"", ""Armagh Observatory (cre)"", ""Burroughs Wellcome (maker)""]"


## 7. Review sample

Head + tail over resolved values, keeping sub-component so inverted and CRF precision read separately.

In [12]:
REVIEW_N = 20

resolved = decisions.filter(pl.col("status") == "resolved")
review = pl.concat([
    resolved.sort("count", descending=True).head(REVIEW_N),
    resolved.sample(min(REVIEW_N, len(resolved)), seed=0),
]).unique("value", maintain_order=True)

review.select("value", "entity_type", "sub_component", "display", "count").write_csv(
    PERSONS_OUT / "person_review_sample.csv")
review.select("value", "entity_type", "sub_component", "display", "count")

value,entity_type,sub_component,display,count
str,str,str,str,u32
"""Petters Ltd""","""organisation""","""router""","""Petters Ltd""",399264
"""Belliss & Morcom Ltd""","""organisation""","""router""","""Belliss & Morcom Ltd""",97500
"""Davey, Paxman & Co. Ltd.""","""organisation""","""router""","""Davey, Paxman & Co. Ltd.""",59293
"""R.A. Lister & Co. Ltd""","""organisation""","""router""","""R.A. Lister & Co. Ltd""",49795
"""National Museums NI""","""organisation""","""crf_corporation""","""National Museums NI""",25454
…,…,…,…,…
"""Genatosan Ltd""","""organisation""","""router""","""Genatosan Ltd""",2
"""Newark Museum and Art Gallery""","""organisation""","""router""","""Newark Museum and Art Gallery""",1
"""Harper, S.""","""person""","""inverted""","""S. Harper""",1
